# Milestone 1 Evaluation Notebook

This notebook runs semantic and BM25 retrieval for the defined query set.
Results are used to populate the markdown evaluation file in results.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.semantic import load_documents, load_faiss_index, semantic_search
from src.bm25 import load_bm25_retriever, bm25_search
from sentence_transformers import SentenceTransformer

DATA_PATH = "../data/processed/processed_data_sample.parquet"
INDEX_PATH = "../data/processed/faiss_index.index"

# Note: HuggingFace warning can be ignored
df, _ = load_documents(DATA_PATH)
index = load_faiss_index(INDEX_PATH)
model = SentenceTransformer("all-MiniLM-L6-v2")
bm25_retriever = load_bm25_retriever("../data/processed/bm25_retriever.pkl")

/Users/clairesaunders/miniforge3/envs/retrieval-app/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 10094.00it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
queries = [
    "blue patio chair cushion",
    "left handed stainless steel garden trowel",
    "6-inch ceramic plant pot with drainage hole",
    "something to make a garden look more colorful and lively",
    "tool for cutting back a large tree",
    "something to keep the deer out of my garden without hurting them",
    "plant suitable for moving outdoors to a patio in spring",
    "best outdoor decoration that won’t fade in the sun",
    "cool gift for someone who loves gardening",
    "what is a good plant to keep outside that won’t die if I forget to water it and doesn’t need much sunlight", 
]

In [3]:
semantic_results = {}

# Run semantic search for all queries
for q in queries:
    results = semantic_search(q, model, index, df, top_k=5)
    semantic_results[q] = results

bm25_results_all = {}

# Run BM25 search for all queries
for q in queries:
    results = bm25_search(bm25_retriever, q, top_k=5)
    bm25_results_all[q] = results

In [4]:
def print_results(query, results):
    print(f"\nQuery: {query}")
    print("-" * 60)
    for r in results:
        print(f"Rank {r['rank']}")
        print(f"Title: {r['product_title']}")
        print(f"Score: {r['score']:.3f}")
        
        print()

# Print Results 
for q in queries:
    print("\n" + "="*80)
    print(f"QUERY: {q}")
    print("="*80)
    
    print("\n--- BM25 Results ---")
    print_results(q, bm25_results_all[q])
    
    print("\n--- Semantic Results ---")
    print_results(q, semantic_results[q])


QUERY: blue patio chair cushion

--- BM25 Results ---

Query: blue patio chair cushion
------------------------------------------------------------
Rank 1
Title: Rocking Chair Cushion Pad, 47IN Indoor Outdoor Navy Blue Rocking Chair Cushions Set Soft Thickened Patio Chaise Lounger Cushion Overstuffed Patio Chair Cushion
Score: 23.801

Rank 2
Title: Rocking Chair Cushion Pad, 2 Pieces Indoor/Outdoor Rocking Chair Cushions Set Indoor/Outdoor Soft Thickened Patio Chaise Lounger Cushion Overstuffed Patio Chair Cushion (Navy Blue)
Score: 23.656

Rank 3
Title: REDCAMP Chaise Lounge Chair Cushion for Patio Furniture, Thicker Soft Comfortable Zero Gravity Chair Pad for Outdoor Indoor Home Office, Blue 65"x21"
Score: 22.973

Rank 4
Title: Comfort Classics Inc. 22W x 44L x 5H Hinge at 24" Sunbrella Outdoor CHANNELED Chair Cushion in Air Blue
Score: 22.658

Rank 5
Title: Pillow Perfect Outdoor/Indoor Basalto Navy Square Corner Chair Cushion, 1 Count (Pack of 1), Blue
Score: 22.212


--- Semantic